# Is it worth walking to the backup dock?

I compare walking to the nearer dock with heading straight to one with a better chance of having an e-bike. I also test what happens if a rider tries the nearer dock first and walks to the backup only if the first is empty. The second experiment asks why the last e-bike at **Totem Park** has been hard to predict.

The two pairs are Bookstore/War Memorial Gym and Totem Park/Orchard Commons. March–October 2026 are rolling evaluation months. They informed earlier work, so the evidence is retrospective. Each decision time represents a possible walk, with no observed rider origin, compliance, reservation or unlock. All inventories in the replay remain as archived.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/prepare_data.py").exists())
sys.path.insert(0, str(root / "notebooks"))
from availability_tools import covered_times, station_decisions, forward_split, predict, warning_weights
from walk_replay import observed_drops, empty_at, risk_at, backup_metrics, weekly_ratio_ci

TZ = "America/Vancouver"
MONTHS = pd.date_range("2026-03-01", "2026-10-01", freq="MS", tz=TZ)
ANCHORS = (1, 5, 10, 15)
SPEEDS = (60, 80, 100)  # metres per minute; 80 is the primary scenario
MIN_GAIN = 0.10         # minimum forecast difference for suggesting a backup
PAIR_IDS = {"Bookstore/Gym": ("2134", "2143"), "Totem/Orchard": ("2117", "2116")}
TOTEM = "2117"

## 1. Walk from the same origin to each dock

These eight fixed points on the campus walking network give two approaches from each side of each pair. I chose the points before inspecting outcomes. Each row contains two pedestrian-route lengths from the same snapped origin, in the order the pair is named. Walk time equals route length divided by assumed speed. The routes are a current map snapshot; historic closures, mobility differences and other docks near each origin are outside this pairwise comparison.

The route matrix was retrieved November 26, 2026. Route data © OpenStreetMap contributors, ODbL. The recorded lengths let this notebook run offline. [Walking route matrix](https://routing.openstreetmap.de/routed-foot/table/v1/foot/-123.2535,49.2630;-123.2488,49.2622;-123.2488,49.2690;-123.2448,49.2675;-123.2537,49.2586;-123.2517,49.2567;-123.2508,49.2630;-123.2467,49.2604;-123.250139,49.265207;-123.248549,49.266579;-123.252100,49.258500;-123.250902,49.259999?sources=0;1;2;3;4;5;6;7;8;9;10;11&destinations=8;9;10;11&annotations=distance,duration&generate_hints=false) · [OpenStreetMap attribution](https://www.openstreetmap.org/copyright)

The matrix returns length along the router's preferred foot path. If walking via the nearer dock gives a shorter route to the farther dock, I use that path for the farther ETA.

In [2]:
routes = pd.DataFrame([
    ("Bookstore/Gym", "southwest approach", 49.262957, -123.253485, 478.8, 696.8),
    ("Bookstore/Gym", "south approach", 49.262085, -123.24872, 460.9, 726.1),
    ("Bookstore/Gym", "north approach", 49.268993, -123.248794, 575.3, 486.6),
    ("Bookstore/Gym", "Western Parkway", 49.267494, -123.244818, 601.9, 545.3),
    ("Totem/Orchard", "west approach", 49.258505, -123.253644, 173.9, 336.2),
    ("Totem/Orchard", "south approach", 49.256689, -123.251789, 360.5, 574.7),
    ("Totem/Orchard", "north approach", 49.262974, -123.250779, 652.6, 450.6),
    ("Totem/Orchard", "east approach", 49.260394, -123.246719, 513.7, 365.7),
], columns=["pair", "origin", "latitude", "longitude", "first_m", "second_m"])
# Walking distances from first to second and from second to first in the same route matrix.
link_m = {"Bookstore/Gym": (245.6, 245.6), "Totem/Orchard": (214.2, 214.2)}
direct_extra = (routes.first_m-routes.second_m).abs()
via_near = [link_m[r.pair][0 if r.first_m <= r.second_m else 1] for r in routes.itertuples()]
display(routes.assign(near_minutes=routes[["first_m","second_m"]].min(axis=1)/80,
                      extra_minutes=np.minimum(direct_extra, via_near)/80).round(2))

,pair,origin,latitude,longitude,first_m,second_m,near_minutes,extra_minutes
0,Bookstore/Gym,southwest approach,49.26,-123.25,478.8,696.8,5.98,2.72
1,Bookstore/Gym,south approach,49.26,-123.25,460.9,726.1,5.76,3.07
2,Bookstore/Gym,north approach,49.27,-123.25,575.3,486.6,6.08,1.11
3,Bookstore/Gym,Western Parkway,49.27,-123.24,601.9,545.3,6.82,0.71
4,Totem/Orchard,west approach,49.26,-123.25,173.9,336.2,2.17,2.03
5,Totem/Orchard,south approach,49.26,-123.25,360.5,574.7,4.51,2.68
6,Totem/Orchard,north approach,49.26,-123.25,652.6,450.6,5.63,2.52
7,Totem/Orchard,east approach,49.26,-123.25,513.7,365.7,4.57,1.85


## 2. What could the rider see before leaving?

Require a positive count at an open or unknown-status dock and a station report at most ten minutes old. A confirmed closure at arrival counts as empty. An unknown count at an open station leaves the outcome unknown. Cover the previous 30 minutes and following 15 minutes with citywide archive activity, rejecting gaps longer than ten minutes. The citywide check cannot detect every individual station outage.

In [3]:
data = root / "data/processed"
inv = pd.read_parquet(data / "ubc_inventory.parquet").sort_values(["station_id","observed_at"])
net = pd.DatetimeIndex(pd.read_parquet(data / "network_observations.parquet").observed_at).sort_values().unique()
end = MONTHS[-1] + pd.offsets.MonthBegin(1)
inv, net = inv.loc[inv.observed_at < end].copy(), net[net < end]
bad = inv.ebikes.lt(0) | inv.ebikes.gt(inv.total_bikes)
inv.loc[bad.fillna(False), "ebikes"] = pd.NA
assert inv.observed_at.isin(net).all() and not inv.duplicated(["station_id","observed_at"]).any()
reports = {s: g for s, g in inv.groupby("station_id")}
names = inv.groupby("station_id").station_name.last()
grid = covered_times(net, interval_min=5, history_min=30, arrival_min=max(ANCHORS), max_gap_min=10)
walks = pd.concat([station_decisions(r, grid, ANCHORS, freshness_min=10) for r in reports.values()],
                  ignore_index=True).sort_values("time").reset_index(drop=True)
for h in ANCHORS:
    count, closed = walks[f"arrival_ebikes_{h}"], walks[f"arrival_closed_{h}"]
    walks[f"empty_{h}"] = (closed | count.eq(0)).where(closed | count.notna())
local = walks.time.dt.tz_convert(TZ)
angle = 2*np.pi*(local.dt.hour + local.dt.minute/60)/24
walks["hour_sin"], walks["hour_cos"] = np.sin(angle), np.cos(angle)
walks["weekend"] = local.dt.dayofweek.ge(5).astype(int)
print(f"{len(walks):,} eligible station starts across {len(reports)} docks")

235,263 eligible station starts across 17 docks


The reference forecast is the same shallow boosted tree used in the school-year replay, with dock, bike counts, local time of day and weekend. Fit four forecast horizons independently so a five-minute destination and a ten-minute destination receive different estimates. Between horizons interpolate their probabilities. Score each month using only labels that have matured before it began.

In [4]:
clock = ["station_id", "ebikes", "total_bikes", "hour_sin", "hour_cos", "weekend"]
def estimator(fields):
    categories = [f for f in fields if f in ("station_id", "class_slot")]
    inputs = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categories),
        ("numbers", "passthrough", [f for f in fields if f not in categories])])
    return make_pipeline(inputs, HistGradientBoostingClassifier(max_depth=3, max_iter=80,
        learning_rate=.07, min_samples_leaf=40, l2_regularization=1, early_stopping=False, random_state=42))

folds = {}
for month in [MONTHS[0] - pd.offsets.MonthBegin(1), *MONTHS]:
    earlier, later = forward_split(walks[["time"]], month, max(ANCHORS))
    assert earlier.time.max() + pd.Timedelta(minutes=max(ANCHORS)) < month
    folds[month] = (earlier.index, later.index)
pair_stations = {s for pair in PAIR_IDS.values() for s in pair}
def monthly_scores(fields, horizon, stations, train_station=None):
    parts = []
    label = f"empty_{horizon}"
    for month, (train_i, test_i) in folds.items():
        if month < MONTHS[0] and stations == {TOTEM}:
            continue  # February calibrates the redirect policy only
        if train_station is not None:
            train_i = train_i[walks.loc[train_i, "station_id"].eq(train_station)]
        train = walks.loc[train_i, fields + [label]].dropna(subset=[label])
        test_i = test_i[walks.loc[test_i, "station_id"].isin(stations)]
        test = walks.loc[test_i, fields]
        score = predict(estimator(fields), fields, train, test, horizon)
        parts.append(pd.Series(score, index=test_i))
    return pd.concat(parts).sort_index()

In [5]:
risk = pd.concat({f"risk_{h}": monthly_scores(clock, h, pair_stations) for h in ANCHORS}, axis=1)
departure = walks.loc[risk.index, ["station_id","time","ebikes","observed_at"]].join(risk)
assert departure[risk.columns].notna().all().all()
print(f"{len(departure):,} scored dock-time observations for the four docks")

30,305 scored dock-time observations for the four docks


## 3. Replay both walks, including a recovery attempt

A comparison requires one advertised e-bike at the nearer dock and at least one at the backup. Join the two departure snapshots first. Read their outcomes afterwards at their exact route ETAs. The recovery ETA is the nearer dock's ETA plus the inter-dock walk. A rider who finds a bike at the first dock stops there. The table shows the number of primary starts, starts with a usable backup, and starts with known paired outcomes.

In [6]:
def pair_cases(route, speed):
    a, b = PAIR_IDS[route.pair]
    first_nearer = route.first_m <= route.second_m
    primary, backup = (a, b) if first_nearer else (b, a)
    near_min, far_min = sorted([route.first_m/speed, route.second_m/speed])
    link_min = link_m[route.pair][0 if first_nearer else 1]/speed
    far_min = min(far_min, near_min + link_min)  # going via the nearer dock is a feasible path
    assert near_min + link_min <= max(ANCHORS) and near_min + link_min + .01 >= far_min
    near = departure.loc[departure.station_id.eq(primary) & departure.ebikes.eq(1)]
    far = departure.loc[departure.station_id.eq(backup)]
    both = near.merge(far, on="time", suffixes=("_near", "_far"), validate="one_to_one")
    both["gain"] = risk_at(both, "near", near_min, ANCHORS) - risk_at(both, "far", far_min, ANCHORS)
    both["near_min"], both["far_min"], both["link_min"] = near_min, far_min, link_min
    for label, station, eta in [("near",primary,near_min), ("far",backup,far_min),
                                ("recovery",backup,near_min + link_min)]:
        deadline = both.time + pd.to_timedelta(eta, unit="min")
        both[label + "_empty"], stamp = empty_at(reports[station], deadline)
        assert (stamp <= deadline).all()
        both[label + "_updated"] = stamp > both["observed_at_near" if label == "near" else "observed_at_far"]
    outcomes = ["near_empty","far_empty","recovery_empty"]
    known = both.dropna(subset=outcomes).copy()
    known[outcomes] = known[outcomes].astype(bool)
    known = known.assign(pair=route.pair, origin=route.origin, primary=primary, speed=speed)
    period = both.time.dt.tz_convert(TZ).dt.tz_localize(None).dt.to_period("M")
    calibration = both[["time","gain"]].assign(pair=route.pair, speed=speed, period=period)
    known = known.loc[known.time >= MONTHS[0]].copy()
    return known, calibration, dict(pair=route.pair, origin=route.origin, primary=names.loc[primary], speed=speed,
                       primary_starts=int(near.time.ge(MONTHS[0]).sum()), usable_backup=int(both.time.ge(MONTHS[0]).sum()), paired_outcomes=len(known))

In [7]:
case_parts, pre_decision, counts = [], [], []
for route in routes.itertuples(index=False):
    for speed in SPEEDS:
        cases, calibration, flow = pair_cases(route, speed)
        pre_decision.append(calibration)
        case_parts.append(cases)
        counts.append(flow)
cases = pd.concat(case_parts, ignore_index=True)
flow = pd.DataFrame(counts)
display(flow.query("speed == 80").drop(columns="speed"))
# Use only the previous month's departure forecasts to set the comparison threshold.
# No arrival outcomes take part, including in the choice of calibration rows.
q = pd.concat(pre_decision).groupby(["pair","speed","period"]).gain.quantile(.80).rename("cutoff").reset_index()
q["calibrated_on"], q["period"] = q.period, q.period + 1
cases["period"] = cases.time.dt.tz_convert(TZ).dt.tz_localize(None).dt.to_period("M")
cases = cases.merge(q, on=["pair","speed","period"], how="left", validate="many_to_one")
has_history = cases.cutoff.notna()
assert (cases.loc[has_history, "calibrated_on"] == cases.loc[has_history, "period"] - 1).all()
# Verify that every missing threshold has no observations in the immediately preceding month.
historical_keys = set(pd.concat(pre_decision)[["pair","speed","period"]].itertuples(index=False, name=None))
absent = cases.loc[~has_history, ["pair","speed","period"]].drop_duplicates()
assert all((pair,speed,period - 1) not in historical_keys for pair,speed,period in absent.itertuples(index=False))
print("Prior-month policy abstains without recent pairs:", list(absent.loc[absent.speed.eq(80), ["pair","period"]].itertuples(index=False, name=None)))
central = cases.loc[cases.speed.eq(80)].copy()
display(central.groupby("pair")[["near_updated","far_updated","recovery_updated"]].mean().round(3))

,pair,origin,primary,primary_starts,usable_backup,paired_outcomes
1,Bookstore/Gym,southwest approach,UBC Bookstore,2824,528,528
4,Bookstore/Gym,south approach,UBC Bookstore,2824,528,528
7,Bookstore/Gym,north approach,War Memorial Gym,3357,552,552
10,Bookstore/Gym,Western Parkway,War Memorial Gym,3357,552,552
13,Totem/Orchard,west approach,Totem Park,2572,381,381
16,Totem/Orchard,south approach,Totem Park,2572,381,381
19,Totem/Orchard,north approach,Orchard Commons,2679,459,459
22,Totem/Orchard,east approach,Orchard Commons,2679,459,459


Prior-month policy abstains without recent pairs: [('Totem/Orchard', Period('2026-07', 'M'))]


,near_updated,far_updated,recovery_updated
pair,,,
Bookstore/Gym,0.271,0.401,0.443
Totem/Orchard,0.208,0.311,0.322


### Does a simple count rule suffice?

Nearest-only is the reference for first attempts. Try two redirect rules set before scoring. **Stock rule** recommends the farther dock when it currently shows at least two e-bikes. **Forecast rule** recommends it when its predicted empty-arrival probability is at least ten percentage points lower at its own ETA. The ten-point rule generated few suggestions in the first replay. As an exploratory sensitivity test, **Prior-month forecast** uses the previous month's 80th percentile of the forecast risk difference for this pair and speed; it also requires a positive advantage. February supplies the first cutoff. When the preceding month has no paired opportunities, this policy makes no suggestion. Only forecasts that would have existed in each earlier month enter calibration. Advice is assumed followed. For decisions without a suggestion the rider takes the near dock.

In [8]:
def apply_rule(frame, rule):
    if rule == "Stock":
        decision = frame.ebikes_far.ge(2)
    elif rule == "Forecast":
        decision = frame.gain.ge(MIN_GAIN)
    elif rule == "Prior-month forecast":
        decision = frame.cutoff.notna() & frame.gain.ge(frame.cutoff) & frame.gain.gt(0)
    else:
        raise KeyError(rule)
    return frame.assign(redirect=decision)
rules = ["Stock", "Forecast", "Prior-month forecast"]
def policy_table(frame, groups):
    rows = []
    for rule in rules:
        ruled = apply_rule(frame, rule)
        for keys, group in ruled.groupby(groups):
            keys = keys if isinstance(keys, tuple) else (keys,)
            rows.append(dict(zip(groups, keys)) | {"rule":rule} | backup_metrics(group).to_dict())
    return pd.DataFrame(rows).set_index(groups + ["rule"])

by_pair = policy_table(central, ["pair"])
display(by_pair[["opportunities","suggestions","suggested_pct","saved_per_100","spoiled_per_100",
                 "near_had_bike_per_100","both_empty_per_100","net_saved_per_100","extra_walk_min_per_100"]].round(2))

,,opportunities,suggestions,suggested_pct,saved_per_100,spoiled_per_100,near_had_bike_per_100,both_empty_per_100,net_saved_per_100,extra_walk_min_per_100
pair,rule,,,,,,,,,
Bookstore/Gym,Stock,2160.0,1192.0,55.19,10.07,2.01,89.93,0.0,8.05,186.28
Totem/Orchard,Stock,1680.0,828.0,49.29,10.02,1.45,89.98,0.0,8.57,225.47
Bookstore/Gym,Forecast,2160.0,1.0,0.05,0.00,0.00,100.00,0.0,0.00,70.75
Totem/Orchard,Forecast,1680.0,18.0,1.07,16.67,0.00,83.33,0.0,16.67,252.50
Bookstore/Gym,Prior-month forecast,2160.0,655.0,30.32,12.37,2.14,87.63,0.0,10.23,168.60
Totem/Orchard,Prior-month forecast,1680.0,364.0,21.67,11.26,1.92,88.74,0.0,9.34,226.15


In [9]:
cutoffs = central.groupby(["period","pair"]).cutoff.first().unstack()
display((100*cutoffs).round(2).rename_axis("evaluation month; cutoff in percentage points"))
print(f"Paired cases among simulated one-bike primary starts: {len(central):,} / "
      f"{flow.loc[flow.speed.eq(80), 'primary_starts'].sum():,}.")

pair,Bookstore/Gym,Totem/Orchard
evaluation month; cutoff in percentage points,,
2026-03,5.18,5.44
2026-04,5.76,5.18
2026-05,5.79,NaN
2026-06,5.62,NaN
2026-07,6.45,NaN
2026-08,6.07,5.80
2026-09,6.03,5.77
2026-10,7.16,7.07


Paired cases among simulated one-bike primary starts: 3,840 / 22,864.


A save means near empty and far usable. A spoiled pickup means near usable and far empty. A needless detour means the nearer dock already had a bike and includes spoiled pickups. Net saves subtract spoiled pickups from saves. Report walking minutes for all recommendations, including unsuccessful ones. Compare net saves per **100 eligible decisions** as well, because the rules can advise different numbers of riders.

In [10]:
display(by_pair[["net_saved_per_100_opportunities","net_pickups_vs_fallback_per_100",
                 "extra_min_vs_fallback_per_100"]].round(2))
directions = policy_table(central.assign(nearer=central.primary.map(names)), ["nearer"])
display(directions[["suggestions","saved_per_100","spoiled_per_100",
                    "net_pickups_vs_fallback_per_100","extra_walk_min_per_100"]].round(1))

,,net_saved_per_100_opportunities,net_pickups_vs_fallback_per_100,extra_min_vs_fallback_per_100
pair,rule,,,
Bookstore/Gym,Stock,4.44,-2.01,155.37
Totem/Orchard,Stock,4.23,-1.45,198.63
Bookstore/Gym,Forecast,0.00,0.00,70.75
Totem/Orchard,Forecast,0.18,0.00,207.88
Bookstore/Gym,Prior-month forecast,3.10,-2.14,130.63
Totem/Orchard,Prior-month forecast,2.02,-1.92,195.99


,,suggestions,saved_per_100,spoiled_per_100,net_pickups_vs_fallback_per_100,extra_walk_min_per_100
nearer,rule,,,,,
Orchard Commons,Stock,492.0,10.8,1.2,-1.2,218.8
Totem Park,Stock,336.0,8.9,1.8,-1.8,235.3
UBC Bookstore,Stock,572.0,14.9,1.4,-1.4,289.7
War Memorial Gym,Stock,620.0,5.6,2.6,-2.6,90.8
Orchard Commons,Forecast,18.0,16.7,0.0,0.0,252.5
Totem Park,Forecast,0.0,NaN,NaN,NaN,NaN
UBC Bookstore,Forecast,0.0,NaN,NaN,NaN,NaN
War Memorial Gym,Forecast,1.0,0.0,0.0,0.0,70.8
Orchard Commons,Prior-month forecast,329.0,11.6,1.8,-1.8,221.7


### How sensitive is the advice to the assumed walk?

Recompute forecasts, outcomes and thresholds at 60 and 100 metres/minute alongside the primary 80 metres/minute. I evaluate the three walking speeds separately because they share the same origins. For uncertainty, resample entire local weeks with all origins kept together so that the same physical inventory change is never treated as a set of independent riders.

At the main speed the prior-month rule made 655 suggestions for Bookstore/Gym and 364 for Totem/Orchard. It improved first-attempt pickups by 10.23 and 9.34 per 100 suggestions, respectively. The near-then-retry policy recovered enough of those misses to finish ahead on eventual pickups and walking time. The fixed ten-point rule rarely fired, so the faster and slower scenarios are important to its interpretation.

In [11]:
sensitivity = policy_table(cases, ["pair","speed"])
display(sensitivity[["suggestions","net_saved_per_100","net_pickups_vs_fallback_per_100",
                     "extra_walk_min_per_100"]].round(1))

suggestions  net_saved_per_100  \
pair          speed rule                                                   
Bookstore/Gym 60    Stock                      1192.0                9.6   
              80    Stock                      1192.0                8.1   
              100   Stock                      1192.0                6.8   
Totem/Orchard 60    Stock                       828.0               10.3   
              80    Stock                       828.0                8.6   
              100   Stock                       828.0                8.0   
Bookstore/Gym 60    Forecast                     90.0               28.9   
              80    Forecast                      1.0                0.0   
              100   Forecast                      0.0                NaN   
Totem/Orchard 60    Forecast                    145.0               12.4   
              80    Forecast                     18.0               16.7   
              100   Forecast                      0.0                NaN   
Bookstore/Gym 60    Prior-month forecast        651.0               11.1   
              80    Prior-month forecast        655.0               10.2   
              100   Prior-month forecast        664.0                9.0   
Totem/Orchard 60    Prior-month forecast        368.0               10.3   
              80    Prior-month forecast        364.0                9.3   
              100   Prior-month forecast        370.0                9.5   

                                          net_pickups_vs_fallback_per_100  \
pair          speed rule                                                    
Bookstore/Gym 60    Stock                                            -2.9   
              80    Stock                                            -2.0   
              100   Stock                                            -1.7   
Totem/Orchard 60    Stock                                            -2.2   
              80    Stock                                            -1.4   
              100   Stock                                            -0.7   
Bookstore/Gym 60    Forecast                                         -3.3   
              80    Forecast                                          0.0   
              100   Forecast                                          NaN   
Totem/Orchard 60    Forecast                                         -2.8   
              80    Forecast                                          0.0   
              100   Forecast                                          NaN   
Bookstore/Gym 60    Prior-month forecast                             -3.1   
              80    Prior-month forecast                             -2.1   
              100   Prior-month forecast                             -1.7   
Totem/Orchard 60    Prior-month forecast                             -2.7   
              80    Prior-month forecast                             -1.9   
              100   Prior-month forecast                             -0.8   

                                          extra_walk_min_per_100  
pair          speed rule                                          
Bookstore/Gym 60    Stock                                  248.4  
              80    Stock                                  186.3  
              100   Stock                                  149.0  
Totem/Orchard 60    Stock                                  300.6  
              80    Stock                                  225.5  
              100   Stock                                  180.4  
Bookstore/Gym 60    Forecast                               227.9  
              80    Forecast                                70.8  
              100   Forecast                                 NaN  
Totem/Orchard 60    Forecast                               310.0  
              80    Forecast                               252.5  
              100   Forecast                                 NaN  
Bookstore/Gym 60    Prior-month forecast  

A quiet dock can carry its original count to the arrival label. Check the forecast policy again on walks where at least one dock, or both docks, published a newer report by their respective direct ETAs. These subsets select busier moments; the full replay remains the primary estimate.

In [12]:
observability = pd.concat([central.assign(reports="all"),
    central.loc[central.near_updated | central.far_updated].assign(reports="at least one newer"),
    central.loc[central.near_updated & central.far_updated].assign(reports="both newer")])
check = policy_table(observability, ["pair","reports"]).xs("Prior-month forecast", level="rule")
display(check[["opportunities","suggestions","net_saved_per_100","net_pickups_vs_fallback_per_100"]].round(2))

opportunities  suggestions  \
pair          reports                                          
Bookstore/Gym all                        2160.0        655.0   
              at least one newer         1190.0        386.0   
              both newer                  262.0        105.0   
Totem/Orchard all                        1680.0        364.0   
              at least one newer          751.0        187.0   
              both newer                  122.0         36.0   

                                  net_saved_per_100  \
pair          reports                                 
Bookstore/Gym all                             10.23   
              at least one newer              17.36   
              both newer                      41.90   
Totem/Orchard all                              9.34   
              at least one newer              18.18   
              both newer                      44.44   

                                  net_pickups_vs_fallback_per_100  
pair          reports                                              
Bookstore/Gym all                                           -2.14  
              at least one newer                            -3.63  
              both newer                                    -1.90  
Totem/Orchard all                                           -1.92  
              at least one newer                            -3.74  
              both newer                                    -8.33

In [13]:
def weeks(times):
    return times.dt.tz_convert(TZ).dt.tz_localize(None).dt.to_period("W-SUN")

bounds = []
for pair, group in central.groupby("pair"):
    delta_near = group.near_empty.astype(int) - group.far_empty.astype(int)
    delta_retry = (group.near_empty & group.recovery_empty).astype(int) - group.far_empty.astype(int)
    stock, model, calibrated = (apply_rule(group, r).redirect.astype(int) for r in rules)
    for comparison, numerator, denominator in [
        ("stock vs near then retry, per 100 suggestions", stock*delta_retry, stock),
        ("prior-month vs near, per 100 suggestions", calibrated*delta_near, calibrated),
        ("prior-month vs near then retry, per 100 suggestions", calibrated*delta_retry, calibrated),
        ("prior-month vs stock, per 100 eligible starts", (calibrated-stock)*delta_near, np.ones(len(group))),
        ("forecast vs near, per 100 suggestions", model*delta_near, model),
        ("forecast vs near then retry, per 100 suggestions", model*delta_retry, model),
        ("forecast vs stock, per 100 eligible starts", (model-stock)*delta_near, np.ones(len(group)))]:
        value, low, high = weekly_ratio_ci(weeks(group.time), numerator, denominator)
        bounds.append([pair, comparison, value, low, high])
display(pd.DataFrame(bounds, columns=["pair","net pickups","estimate","lower 95%","upper 95%"]).round(2))
print("Intervals omitted when fewer than five contributing weeks have weight in a denominator.")

,pair,net pickups,estimate,lower 95%,upper 95%
0,Bookstore/Gym,"stock vs near then retry, per 100 suggestions",-2.01,-3.41,-0.90
1,Bookstore/Gym,"prior-month vs near, per 100 suggestions",10.23,4.65,14.96
2,Bookstore/Gym,"prior-month vs near then retry, per 100 sugges...",-2.14,-4.49,-0.52
3,Bookstore/Gym,"prior-month vs stock, per 100 eligible starts",-1.34,-2.70,-0.29
4,Bookstore/Gym,"forecast vs near, per 100 suggestions",0.00,NaN,NaN
5,Bookstore/Gym,"forecast vs near then retry, per 100 suggestions",0.00,NaN,NaN
6,Bookstore/Gym,"forecast vs stock, per 100 eligible starts",-4.44,-6.10,-2.61
7,Totem/Orchard,"stock vs near then retry, per 100 suggestions",-1.45,-2.91,-0.31
8,Totem/Orchard,"prior-month vs near, per 100 suggestions",9.34,5.12,14.78
9,Totem/Orchard,"prior-month vs near then retry, per 100 sugges...",-1.92,-5.08,0.00


Intervals omitted when fewer than five contributing weeks have weight in a denominator.


## 4. How predictable are the sharp changes at Totem?

Focus on ten-minute one-bike walks at Totem. Match every feature comparison on identical station-times and outcomes. Train on earlier campus observations so other docks can contribute evidence about recurring patterns. Keep the tree, folds, warning budget and primary outcome fixed.

The primary comparison is the combination of school-weekday timing and observed decreases at neighbouring docks against the original clock. Additional controls isolate the time window, individual feature groups, and whether more frequent reporting itself is informative.

In [14]:
school = local.dt.month.ge(9) | local.dt.month.le(4)
gate = school & local.dt.dayofweek.lt(5) & local.dt.hour.between(8, 17)
walks["class_window"] = gate.astype(int)
walks["class_slot"] = local.dt.minute.astype(str).where(gate, "off")
phase = local.dt.minute % 30
walks["half_hour_distance"] = np.where(gate, np.minimum(phase, 30-phase), -1)
# This is a calendar proxy. The archive has no course timetable or attendance data.
timing = ["class_window", "class_slot", "half_hour_distance"]
for lag in (5, 15):
    walks[f"own_change_{lag}"] = (walks.ebikes - walks[f"ebikes_{lag}m_ago"]).astype(float)
own_history = ["own_change_5","own_change_15","reports_15","age_min"]

### Which neighbours could the model see?

Define neighbours from station coordinates, with the focal dock excluded. Use the three nearest campus sites known by each decision time (all 17 were observed before the evaluation began). For a given neighbour, a decrease counts only when both readings were open or had unknown rental status, their counts were known, they were at most fifteen minutes apart, and the citywide collector covered the interval. The event time is when the later report arrived. Decreases can include staff activity.

Summarize the two disjoint windows (0–15 and 15–30 minutes before departure). A silent neighbour contributes no evidence of a decrease. Count neighbours with comparable reports; a completely unobserved window gets a missing decrease value.

In [15]:
coords = inv.groupby("station_id").first()[["latitude","longitude"]]
first_report = inv.groupby("station_id").observed_at.min()
lat, lon = np.radians(coords.latitude.to_numpy()), np.radians(coords.longitude.to_numpy())
a = np.sin((lat[:,None]-lat)/2)**2 + np.cos(lat[:,None])*np.cos(lat)*np.sin((lon[:,None]-lon)/2)**2
dist = pd.DataFrame(12742000*np.arcsin(np.sqrt(a)), index=coords.index, columns=coords.index)
close = dist.loc[TOTEM].drop(TOTEM).nsmallest(3)
display(pd.DataFrame({"neighbour":close.index.map(names), "straight_line_m":close.round()}))

,neighbour,straight_line_m
station_id,,
2116,Orchard Commons,188.0
2127,Thunderbird & Thunderbird,335.0
2114,West Mall Swing Space Building,533.0


In [16]:
segment = net.to_series().diff().gt(pd.Timedelta(minutes=10)).cumsum()
times = pd.DatetimeIndex(grid.time)
signals = {s: observed_drops(r, segment, times) for s,r in reports.items()}
locations = first_report.sort_values()
# The neighbour set can change when a historical station first appears.
eras = locations.drop_duplicates().sort_values().tolist()
pieces = []
for start, stop in zip(eras, eras[1:] + [end]):
    available = locations.index[locations.le(start)]
    mask = (times >= start) & (times < stop)
    if not mask.any() or len(available) < 2:
        continue
    for station in available:
        nearest = dist.loc[station, available].drop(station).nsmallest(3).index
        context = pd.DataFrame({"time":times[mask], "station_id":station})
        for period in ("recent","prior"):
            seen = sum(signals[s].loc[mask, period + "_seen"].to_numpy(dtype=int) for s in nearest)
            loss = sum(signals[s].loc[mask, period].to_numpy() for s in nearest)
            context["seen_" + period] = seen
            context["loss_" + period] = np.where(seen > 0, loss, np.nan)
        pieces.append(context)
signals_at_start = pd.concat(pieces, ignore_index=True)
keys = ["time","station_id"]
# Preserve the row indices used by the original time splits.
feature_columns = ["seen_recent","seen_prior","loss_recent","loss_prior"]
lookup = signals_at_start.set_index(keys)[feature_columns]
walks[feature_columns] = lookup.reindex(pd.MultiIndex.from_frame(walks[keys])).to_numpy()
totem_test = departure.index[departure.station_id.eq(TOTEM) & departure.ebikes.eq(1) & departure.time.ge(MONTHS[0])]
eligible = walks.loc[totem_test].dropna(subset=["empty_10"])
display(eligible[feature_columns].agg(["count","mean"]).round(2))
print(f"{len(eligible):,} Totem decisions; observable recent neighbour decreases on {eligible.loss_recent.notna().mean():.1%}.")

,seen_recent,seen_prior,loss_recent,loss_prior
count,2572.00,2572.00,962.00,963.00
mean,0.47,0.47,0.77,0.79


2,572 Totem decisions; observable recent neighbour decreases on 37.4%.


### Do quarter-hour timing and neighbouring decreases add predictive signal?

Use a fixed 20% warning budget inside each month for this ranking comparison. Tied predictions get equal fractional credit. Report counts, precision, recall, average precision and Brier score. Probability errors and ranking can respond differently. As in earlier pilots, a live cutoff would need to be set before deployment.

In [17]:
from sklearn.metrics import average_precision_score
observed = eligible[["time","empty_10","class_window"]].copy()
observed["y"] = observed.empty_10.astype(int)
observed["month"] = observed.time.dt.tz_convert(TZ).dt.strftime("%Y-%m")
observed["school"] = observed.time.dt.tz_convert(TZ).dt.month.isin([1,2,3,4,9,10,11,12])
scorecards = {}
def score_variant(name, fields=None, train_station=None):
    score = risk["risk_10"] if fields is None else monthly_scores(fields,10,{TOTEM}, train_station)
    df = observed.assign(score=score.reindex(observed.index))
    assert df.score.notna().all()
    df["w"] = df.groupby("month").score.transform(lambda x: warning_weights(x, .20))
    df["hit"] = df.w*df.y
    scorecards[name] = df
    return {"walks":len(df), "empty":int(df.y.sum()), "precision_pct":100*df.hit.sum()/df.w.sum(),
            "recall_pct":100*df.hit.sum()/df.y.sum(), "AP":average_precision_score(df.y,df.score),
            "Brier":((df.score-df.y)**2).mean()}

summary = {}
summary["Clock"] = score_variant("Clock")
summary["School weekday window"] = score_variant("School weekday window", clock + ["class_window"])
summary["Class timing"] = score_variant("Class timing", clock + timing)
display(pd.DataFrame(summary).T.round(4))

,walks,empty,precision_pct,recall_pct,AP,Brier
Clock,2572.0,363.0,14.8668,21.0674,0.1679,0.1222
School weekday window,2572.0,363.0,15.5187,21.9912,0.1703,0.1219
Class timing,2572.0,363.0,16.3850,23.2189,0.1767,0.1218


In [18]:
coverage = ["seen_recent","seen_prior"]
losses = ["loss_recent","loss_prior"]
summary["Neighbour coverage"] = score_variant("Neighbour coverage", clock + coverage)
summary["Neighbour losses"] = score_variant("Neighbour losses", clock + coverage + losses)
display(pd.DataFrame(summary).T.loc[["Clock","Neighbour coverage","Neighbour losses"]].round(4))

,walks,empty,precision_pct,recall_pct,AP,Brier
Clock,2572.0,363.0,14.8668,21.0674,0.1679,0.1222
Neighbour coverage,2572.0,363.0,15.2419,21.5990,0.1687,0.1219
Neighbour losses,2572.0,363.0,14.7091,20.8439,0.1731,0.1219


In [19]:
primary = "Timing + neighbours"
summary[primary] = score_variant(primary, clock + timing + coverage + losses)
summary["Own history"] = score_variant("Own history", clock + own_history)
summary["Own + timing + neighbours"] = score_variant("Own + timing + neighbours", clock + own_history + timing + coverage + losses)
display(pd.DataFrame(summary).T.round(4))

,walks,empty,precision_pct,recall_pct,AP,Brier
Clock,2572.0,363.0,14.8668,21.0674,0.1679,0.1222
School weekday window,2572.0,363.0,15.5187,21.9912,0.1703,0.1219
Class timing,2572.0,363.0,16.3850,23.2189,0.1767,0.1218
Neighbour coverage,2572.0,363.0,15.2419,21.5990,0.1687,0.1219
Neighbour losses,2572.0,363.0,14.7091,20.8439,0.1731,0.1219
Timing + neighbours,2572.0,363.0,16.6861,23.6455,0.1792,0.1215
Own history,2572.0,363.0,15.6104,22.1212,0.1715,0.1220
Own + timing + neighbours,2572.0,363.0,16.6148,23.5445,0.1792,0.1216


### Could pooling all 17 docks conceal a pattern unique to Totem?

Fit the clock and combined model on earlier Totem walks alone. Keep their tree depth, leaf size and other settings fixed, then score the same walks. This compares two training populations without changing the outcome or the inputs within each pair.

On 2,572 one-bike walks, the pooled clock correctly identified 14.87 empty arrivals per 100 warnings. Calendar timing alone reached 16.39; the neighbour-loss variant reached 14.71. A comparable transition at any neighbour was observable in just 37.4% of recent windows. The combined model reached 16.69. The station-only fit below checks whether Totem's own history is enough to learn a more useful pattern.

In [20]:
summary["Totem-trained clock"] = score_variant("Totem-trained clock", clock, TOTEM)
summary["Totem-trained combined"] = score_variant("Totem-trained combined", clock + timing + coverage + losses, TOTEM)
display(pd.DataFrame(summary).T.loc[["Clock",primary,"Totem-trained clock","Totem-trained combined"]].round(4))

,walks,empty,precision_pct,recall_pct,AP,Brier
Clock,2572.0,363.0,14.8668,21.0674,0.1679,0.1222
Timing + neighbours,2572.0,363.0,16.6861,23.6455,0.1792,0.1215
Totem-trained clock,2572.0,363.0,16.6500,23.5944,0.1747,0.1218
Totem-trained combined,2572.0,363.0,19.3596,27.4341,0.2030,0.1205


### Does the result persist across months and school-day hours?

Keep the same monthly warning assignments for the school/summer and school-day slices. Report how many warnings and empty arrivals fall in each slice. Week-block uncertainty intervals pair both models on exactly the same Totem walks. The combined variant is the primary test; the others are diagnostic.

In [21]:
comparison = pd.concat({name:scorecards[name] for name in ["Clock",primary]}, names=["model"]).reset_index(level=0)
def subset_scores(frame, by):
    g = frame.groupby(by).agg(walks=("y","size"), empty=("y","sum"), warnings=("w","sum"), hits=("hit","sum"))
    return g.assign(precision_pct=100*g.hits/g.warnings).drop(columns="hits")
display(subset_scores(comparison, ["month","model"]).precision_pct.unstack().round(2))
slices = pd.concat([comparison.assign(period="all"),comparison.loc[comparison.school].assign(period="school"),
                    comparison.loc[~comparison.school].assign(period="summer"),
                    comparison.loc[comparison.class_window.eq(1)].assign(period="school weekdays 08–18")])
display(subset_scores(slices,["period","model"]).round(2))

model,Clock,Timing + neighbours
month,,
2026-03,10.83,11.56
2026-04,16.37,20.05
2026-05,0.00,0.00
2026-06,5.33,7.26
2026-07,3.25,6.71
2026-08,13.81,13.81
2026-09,15.04,18.80
2026-10,23.92,24.03


walks  empty  warnings  \
period                model                                         
all                   Clock                 2572    363     514.4   
                      Timing + neighbours   2572    363     514.4   
school                Clock                 2095    335     419.0   
                      Timing + neighbours   2095    335     419.0   
school weekdays 08–18 Clock                  982    179     373.6   
                      Timing + neighbours    982    179     385.6   
summer                Clock                  477     28      95.4   
                      Timing + neighbours    477     28      95.4   

                                           precision_pct  
period                model                               
all                   Clock                        14.87  
                      Timing + neighbours          16.69  
school                Clock                        16.98  
                      Timing + neighbours          18.85  
school weekdays 08–18 Clock                        16.68  
                      Timing + neighbours          17.89  
summer                Clock                         5.57  
                      Timing + neighbours           7.16

In [22]:
paired = []
for candidate, reference in [(primary,"Clock"), ("Class timing","School weekday window"),
                             ("Neighbour losses","Neighbour coverage"),("Own + timing + neighbours","Own history"),
                             ("Totem-trained combined","Totem-trained clock")]:
    x, y = scorecards[candidate], scorecards[reference]
    assert x.index.equals(y.index)
    estimate, low, high = weekly_ratio_ci(weeks(x.time),x.hit,x.w,second=(y.hit,y.w))
    paired.append([candidate, reference, estimate, low, high])
display(pd.DataFrame(paired, columns=["candidate","reference","precision gain (pp)","lower 95%","upper 95%"]).round(2))

,candidate,reference,precision gain (pp),lower 95%,upper 95%
0,Timing + neighbours,Clock,1.82,0.22,3.51
1,Class timing,School weekday window,0.87,-0.47,2.37
2,Neighbour losses,Neighbour coverage,-0.53,-2.13,1.10
3,Own + timing + neighbours,Own history,1.00,-0.76,2.92
4,Totem-trained combined,Totem-trained clock,2.71,0.59,4.52


## What should the app do?

**For these two nearby pairs, show the fallback and keep the nearer dock first.** At 80 metres/minute, the previous-month cutoff made 1,019 redirect suggestions across 3,840 eligible origin-time scenarios. About 88 in every 100 redirects passed a nearer dock that still had an e-bike. Relative to trying the nearer dock and stopping after a miss, the redirect gained 10.23 first-attempt pickups per 100 suggestions at Bookstore/Gym and 9.34 at Totem/Orchard. Relative to trying the near dock and walking to the backup if necessary, the redirect **lost 2.14 and 1.92 eventual pickups** per 100 suggestions and added **131 and 196 minutes of walking**, respectively. The stock rule issued more suggestions and also lost to trying the nearer dock first on eventual pickups. Riders who would give up after the first miss face a different trade-off.

**Totem's own history was informative.** The model trained only on Totem reached 19.36 correct warnings per 100 using school-day timing and neighbouring inventory, versus 16.65 with its clock alone. The paired-week interval for the gain was +0.59 to +4.52 percentage points. The preselected pooled comparison gained 1.82 points (+0.22 to +3.51). The neighbour-loss-only model did not establish a gain over its reporting-coverage control. These experiments support a modest risk signal. Whether students were leaving class is unknown.

Eligible scenarios required a recent positive report at both docks; they accounted for 3,840 of 22,864 simulated nearest-dock one-bike starts. Repeated origins can see the same inventory event, and quiet stations can retain an earlier archive reading. Changing the route online would itself change inventory. The experiment measures reported availability and modeled walking time; rider compliance and successful unlocks still need field observations.